# 03 Train Churn Model

Phase 4b: churn classifier for Emberline Studios players. Each row is a player active in the 30 days before a month-end observation date; the label is no session in the next 30 days. Features cover the 30-day engagement trend, toxicity received, matchmaking quality and spend. Scored at as_of_date (2026-12-31) into `ml_churn_player_scores` with `churn_risk` (0/1) and `churn_risk_score`.

In [0]:
from pyspark.sql import functions as F

CATALOG = "mg_aws_serverless_workspace_catalog"
SCHEMA = "game_cx_players"
FQ = f"{CATALOG}.{SCHEMA}"
AS_OF_DATE = "2026-12-31"
TRAIN_END = "2026-08-31"     # train on observation dates up to here
VALID_OBS = "2026-09-30"     # threshold chosen on this observation date
TEST_OBS = "2026-11-30"      # held-out test (label window = December 2026)
EXPERIMENT = "/Users/mayur.gupta@databricks.com/game-cx-players-churn"
MODEL_NAME = f"{FQ}.churn_player_classifier"
GT = f"/Volumes/{CATALOG}/{SCHEMA}/ground_truth"
print(FQ, TRAIN_END, VALID_OBS, TEST_OBS, AS_OF_DATE)

In [0]:
def window_sum(col, obs, lo, hi):
    """Sum `col` over rows whose `d` falls in (obs - lo, obs - hi] days."""
    return F.sum(F.when((F.col("d") > F.date_sub(obs, lo)) & (F.col("d") <= F.date_sub(obs, hi)), F.col(col)).otherwise(0))


def build_churn_features():
    """Build one row per (player, month-end observation date) for players active in the prior 30 days.

    Sessions, purchases and toxicity signals are first rolled up to player-day, then joined to
    observation dates with a 60-day look-back (features) and 30-day look-ahead (label).
    Observation 2026-10-31 is skipped so the training labels never overlap the test window.

    Returns:
        Spark DataFrame with features, `label` (1 = no session in next 30 days, NULL for as_of_date) and `role`.
    """
    obs = (spark.range(1)
           .select(F.explode(F.sequence(F.lit("2025-03-01").cast("date"), F.lit("2026-12-01").cast("date"),
                                        F.expr("INTERVAL 1 MONTH"))).alias("m"))
           .select(F.last_day("m").alias("obs"))
           .withColumn("role", F.when(F.col("obs") <= TRAIN_END, "train").when(F.col("obs") == VALID_OBS, "valid")
                       .when(F.col("obs") == TEST_OBS, "test").when(F.col("obs") == AS_OF_DATE, "score"))
           .filter("role IS NOT NULL"))

    sess = (spark.table(f"{FQ}.gold_fact_session").groupBy("player_id", F.col("session_date").alias("d"))
            .agg(F.count("*").alias("n_sess"), F.sum("duration_min").alias("minutes"),
                 F.sum("matchmaking_quality").alias("mmq_sum"), F.count("matchmaking_quality").alias("mmq_n"),
                 F.sum("ping_ms").alias("ping_sum"), F.lit(1).alias("active_day")))
    s = (sess.join(obs, (F.col("d") > F.date_sub("obs", 60)) & (F.col("d") <= F.date_add("obs", 30)))
         .groupBy("player_id", "obs", "role")
         .agg(window_sum("n_sess", "obs", 30, 0).alias("sessions_30d"),
              window_sum("n_sess", "obs", 60, 30).alias("sessions_prev_30d"),
              window_sum("minutes", "obs", 30, 0).alias("minutes_30d"),
              window_sum("active_day", "obs", 30, 0).alias("active_days_30d"),
              window_sum("mmq_sum", "obs", 30, 0).alias("mmq_sum"), window_sum("mmq_n", "obs", 30, 0).alias("mmq_n"),
              window_sum("ping_sum", "obs", 30, 0).alias("ping_sum"),
              F.max(F.when(F.col("d") <= F.col("obs"), F.col("d"))).alias("last_session"),
              F.sum(F.when(F.col("d") > F.col("obs"), F.col("n_sess")).otherwise(0)).alias("future_sessions"))
         .filter("sessions_30d > 0"))

    txn = (spark.table(f"{FQ}.silver_transactions").groupBy("player_id", F.col("txn_date").alias("d"))
           .agg(F.sum("amount_usd").alias("usd")))
    t = (txn.join(obs, F.col("d") <= F.col("obs")).groupBy("player_id", "obs")
         .agg(F.sum("usd").alias("lifetime_spend_usd"), window_sum("usd", "obs", 30, 0).alias("spend_30d")))

    reports = spark.table(f"{FQ}.silver_player_reports")
    evidence = reports.select(F.col("evidence_message_id").alias("message_id")).where("message_id IS NOT NULL").distinct()
    tox_recv = (spark.table(f"{FQ}.silver_chat_messages").join(evidence, "message_id")
                .groupBy(F.col("recipient_player_id").alias("player_id"), F.col("sent_date").alias("d"))
                .agg(F.count("*").alias("n_tox")))
    filed = reports.groupBy(F.col("reporter_player_id").alias("player_id"), F.col("report_date").alias("d")).agg(F.count("*").alias("n_filed"))
    tox = (tox_recv.join(filed, ["player_id", "d"], "full").fillna(0)
           .join(obs, (F.col("d") > F.date_sub("obs", 30)) & (F.col("d") <= F.col("obs")))
           .groupBy("player_id", "obs").agg(F.sum("n_tox").alias("toxic_msgs_received_30d"), F.sum("n_filed").alias("reports_filed_30d")))

    players = spark.table(f"{FQ}.silver_players").select("player_id", "registration_date")
    return (s.join(t, ["player_id", "obs"], "left").join(tox, ["player_id", "obs"], "left").join(players, "player_id", "left")
            .fillna(0, ["lifetime_spend_usd", "spend_30d", "toxic_msgs_received_30d", "reports_filed_30d"])
            .select("player_id", "obs", "role", "sessions_30d", "sessions_prev_30d", "minutes_30d", "active_days_30d",
                    ((F.col("sessions_30d") - F.col("sessions_prev_30d")) / (F.col("sessions_prev_30d") + 1)).alias("engagement_trend"),
                    F.datediff("obs", "last_session").alias("days_since_last_session"),
                    F.try_divide("mmq_sum", "mmq_n").alias("matchmaking_quality_30d"),
                    F.try_divide("ping_sum", "sessions_30d").alias("ping_ms_30d"),
                    "toxic_msgs_received_30d", "reports_filed_30d", "spend_30d", "lifetime_spend_usd",
                    F.datediff("obs", "registration_date").alias("account_age_days"),
                    F.when(F.col("role") == "score", F.lit(None)).otherwise((F.col("future_sessions") == 0).cast("int")).alias("label")))


build_churn_features().write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.ml_feat_churn_player")
feat = spark.table(f"{FQ}.ml_feat_churn_player")
display(feat.groupBy("role").agg(F.count("*").alias("rows"), F.countDistinct("obs").alias("obs_dates"),
                                 F.round(F.avg("label"), 4).alias("churn_rate"),
                                 F.round(F.avg("toxic_msgs_received_30d"), 3).alias("avg_tox_recv")).orderBy("role"))

In [0]:
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score

# Features are aggregated in PySpark from 7M sessions. The modelling table is ~490K rows x 14 numeric
# columns, small enough for pandas + scikit-learn, which is faster to iterate on than Spark ML here.
FEATURES = ["sessions_30d", "sessions_prev_30d", "minutes_30d", "active_days_30d", "engagement_trend",
            "days_since_last_session", "matchmaking_quality_30d", "ping_ms_30d", "toxic_msgs_received_30d",
            "reports_filed_30d", "spend_30d", "lifetime_spend_usd", "account_age_days"]
pdf = feat.toPandas()
train, valid, test = (pdf[pdf.role == r] for r in ("train", "valid", "test"))


def top_decile_lift(y, score):
    """Churn rate among the top 10% of scores divided by the overall churn rate."""
    k = max(1, int(len(score) * 0.10))
    return float(y[np.argsort(-score)[:k]].mean() / max(1e-9, y.mean()))


def best_f1_threshold(y, score):
    """Return the score threshold that maximises F1 on a validation set."""
    grid = np.quantile(score, np.linspace(0.50, 0.995, 100))
    return float(max(grid, key=lambda t: f1_score(y, score >= t)))


mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name="hgb_churn_player") as run:
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_leaf_nodes=31, random_state=42)
    model.fit(train[FEATURES], train.label)
    THRESHOLD = best_f1_threshold(valid.label.to_numpy(), model.predict_proba(valid[FEATURES])[:, 1])
    p = model.predict_proba(test[FEATURES])[:, 1]
    y, yhat = test.label.to_numpy(), (p >= THRESHOLD).astype(int)
    metrics = {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
               "top_decile_lift": top_decile_lift(y, p), "test_churn_rate": float(y.mean()),
               "precision_at_threshold": precision_score(y, yhat), "recall_at_threshold": recall_score(y, yhat),
               "flag_rate_at_threshold": float(yhat.mean()), "threshold": THRESHOLD}
    mlflow.log_params({"train_rows": len(train), "valid_rows": len(valid), "test_rows": len(test),
                       "train_end": TRAIN_END, "valid_obs": VALID_OBS, "test_obs": TEST_OBS, "features": ",".join(FEATURES)})
    mlflow.log_metrics(metrics)
    sig = infer_signature(test[FEATURES].head(100), p[:100])
    mlflow.sklearn.log_model(model, name="model", signature=sig, input_example=test[FEATURES].head(5),
                             registered_model_name=MODEL_NAME)
    RUN_ID = run.info.run_id
print(RUN_ID)
display(pd.DataFrame({"metric": list(metrics), "value": list(metrics.values())}))

In [0]:
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'") if v.run_id == RUN_ID)
client.set_registered_model_alias(MODEL_NAME, "champion", version)
print(f"{MODEL_NAME} v{version} -> @champion")

In [0]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
active = pdf[pdf.role == "score"][["player_id"] + FEATURES].copy()
active["churn_risk_score"] = champion.predict_proba(active[FEATURES])[:, 1]
scored = spark.createDataFrame(active[["player_id", "churn_risk_score"]])

# Every player gets a row. Players with no session in the last 30 days have already lapsed:
# they are not scored and not counted as at-risk (churn_risk = 0).
last_seen = spark.table(f"{FQ}.gold_fact_session").groupBy("player_id").agg(F.max("session_date").alias("last_session_date"))
scores = (spark.table(f"{FQ}.silver_players").select("player_id")
          .join(scored, "player_id", "left").join(last_seen, "player_id", "left")
          .withColumn("player_status", F.when(F.col("churn_risk_score").isNotNull(), "active")
                      .when(F.col("last_session_date").isNull(), "never_played").otherwise("lapsed"))
          .withColumn("churn_risk", (F.coalesce(F.col("churn_risk_score"), F.lit(0.0)) >= F.lit(THRESHOLD)).cast("int"))
          .withColumn("model_version", F.lit(int(version)))
          .withColumn("scored_as_of", F.lit(AS_OF_DATE).cast("date"))
          .select("player_id", "player_status", "last_session_date", "churn_risk_score", "churn_risk", "model_version", "scored_as_of"))
(scores.write.mode("overwrite").option("overwriteSchema", "true")
       .option("description", "Churn score per player at as_of_date 2026-12-31 from churn_player_classifier@champion. churn_risk = 1 (at-risk) when churn_risk_score >= the F1-optimal threshold chosen on the 2026-09-30 validation month. Lapsed and never-played players are not scored.")
       .saveAsTable(f"{FQ}.ml_churn_player_scores"))
display(spark.sql(f"""
SELECT
  player_status,
  COUNT(*) AS players,
  SUM(churn_risk) AS at_risk_players,
  ROUND(AVG(churn_risk_score), 4) AS avg_score
FROM {FQ}.ml_churn_player_scores
GROUP BY ALL
ORDER BY player_status
"""))

In [0]:
gt = spark.read.parquet(f"{GT}/gt_players").select("player_id", "is_victim", "has_bad_matchmaking", "is_churned")
chk = spark.table(f"{FQ}.ml_churn_player_scores").filter("player_status = 'active'").join(gt, "player_id")
for flag in ["is_victim", "has_bad_matchmaking", "is_churned"]:
    display(chk.groupBy(flag).agg(F.count("*").alias("active_players"), F.round(F.avg("churn_risk"), 4).alias("at_risk_rate"),
                                  F.round(F.avg("churn_risk_score"), 4).alias("avg_score")).orderBy(flag))